# OCR Pipeline: Stamp Removal + Text Extraction
**Local Jupyter Notebook**

This notebook executes a complete 3-phase document processing pipeline:
- **Phase 3A**: Stamp removal via chromatic pixel separation + intelligent inpainting
- **Phase 4**: OCR text extraction using EasyOCR with spelling correction
- **Phase 5** (optional): Evaluate OCR quality using CER/WER metrics

**Run cells top-to-bottom.** Each cell depends on the previous one.

---
## Table of Contents
1. Configure paths
2. Import pipeline functions
3. Load document image
4. Phase 3A: Stamp removal (with evidence grid)
5. Phase 4: OCR extraction
6. **[NEW]** Phase 4: Color-coded detection visualization
7. Consolidate results + metrics dashboard
8. Phase 5: Evaluation (optional)
9. Summary

## Cell 1: Configuration

In [1]:
from pathlib import Path
import os
import sys

# === CONFIGURATION ===
# Update this path to point to your document image
INPUT_IMAGE = r"C:\Users\manue\Desktop\final_vision\output\prueba_completa\01_imagen_original.png"

# Repository root (adjust if running from different location)
REPO_ROOT = Path(os.getcwd())  # Assumes notebook is in /notebooks
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

print(f"[OK] Repository root: {REPO_ROOT}")
print(f"[OK] Input image: {INPUT_IMAGE}")

# Verify file exists
if not Path(INPUT_IMAGE).exists():
    print(f"[ERROR] Image not found: {INPUT_IMAGE}")
    print(f"[INFO] Please update INPUT_IMAGE in Cell 1")
else:
    print(f"[OK] Image file exists")

# Create output directory
OUTPUT_ROOT = Path(REPO_ROOT) / "output" / "pipeline_local"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[OK] Output directory: {OUTPUT_ROOT}")

[OK] Repository root: c:\Users\manue\Desktop\final_vision
[OK] Input image: C:\Users\manue\Desktop\final_vision\output\prueba_completa\01_imagen_original.png
[OK] Image file exists
[OK] Output directory: c:\Users\manue\Desktop\final_vision\output\pipeline_local


## Cell 2: Import Pipeline Functions

In [2]:
# Add repo to path
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import pipeline API
from pipeline import (
    run_phase_3a,
    run_phase_4,
    consolidate_ocr_results,
    run_evaluation,
    run_full_pipeline,
)

print("[OK] Pipeline functions imported successfully")
print("\nAvailable functions:")
print("  - run_phase_3a() ........... Stamp removal")
print("  - run_phase_4() ........... OCR extraction")
print("  - consolidate_ocr_results() . Extract text + metrics")
print("  - run_evaluation() ........ CER/WER evaluation")
print("  - run_full_pipeline() ..... All phases together")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\manue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\manue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\traitlets\config\application.py", line 1075, in launch

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## Cell 3: Load Document Image

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Verify and load image
img_path = Path(INPUT_IMAGE)

if not img_path.exists():
    print(f"[ERROR] Image not found: {INPUT_IMAGE}")
    print(f"[INFO] Update INPUT_IMAGE in Cell 1 and re-run")
else:
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"[ERROR] Failed to load image: {INPUT_IMAGE}")
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(12, 10))
        plt.imshow(img_rgb)
        plt.title(f"Input Document: {img_path.name}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        
        print(f"[OK] Image loaded successfully")
        print(f"  File: {img_path.name}")
        print(f"  Size: {img.shape[1]}x{img.shape[0]} px")

## Cell 4: Phase 3A - Stamp Removal (with Evidence Grid)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

PHASE_3A_OUTPUT = str(OUTPUT_ROOT / "phase_3a")

print("Running Phase 3A: Stamp Removal...")
print("[*] This may take 30-60 seconds depending on image size\n")

result_3a = run_phase_3a(
    input_image_path=INPUT_IMAGE,
    output_dir=PHASE_3A_OUTPUT,
    block_size=20,
    inpainting_method="hybrid",
    save_intermediate=True,
)

if result_3a["success"]:
    CLEANED_IMAGE = result_3a["output_image"]
    print(f"\n[OK] Phase 3A completed successfully")
    print(f"  Cleaned image: {CLEANED_IMAGE}")
    print(f"  Pixels inpainted: {result_3a.get('pixels_inpainted', 'N/A')}")
    print(f"  Inpainting method: {result_3a.get('inpainting_method', 'N/A')}")

    # === EVIDENCE GRID: Load and display intermediate images ===
    output_dir_path = Path(PHASE_3A_OUTPUT)
    intermediate_images = sorted(output_dir_path.glob("0*.png"))
    
    if len(intermediate_images) > 0:
        print(f"\n[*] Loading {len(intermediate_images)} intermediate images...")
        
        # Load up to 6 key images for the grid
        evidence_images = []
        evidence_labels = []
        
        # Map filenames to meaningful labels
        name_mapping = {
            "01": "Original Image",
            "02": "Pixel Classification",
            "03": "H-Sweep Classification",
            "04": "Block Grid Overlay",
            "05": "Final Mask",
            "06": "Before/After",
        }
        
        for i, img_path in enumerate(intermediate_images[:6]):
            try:
                img = cv2.imread(str(img_path))
                if img is not None:
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    evidence_images.append(img_rgb)
                    
                    # Extract number from filename (e.g., "01_..." -> "01")
                    fname = img_path.stem  # e.g., "01_original"
                    prefix = fname[:2]  # e.g., "01"
                    label = name_mapping.get(prefix, f"Step {i+1}")
                    evidence_labels.append(label)
                    print(f"  [{i+1}/6] Loaded: {img_path.name}")
            except Exception as e:
                pass
        
        # Display evidence grid (2 rows x 3 cols)
        if len(evidence_images) >= 4:
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            axes = axes.flatten()
            
            for idx, (img, label) in enumerate(zip(evidence_images, evidence_labels)):
                axes[idx].imshow(img)
                axes[idx].set_title(label, fontsize=11, fontweight="bold")
                axes[idx].axis("off")
            
            # Hide unused subplots
            for idx in range(len(evidence_images), 6):
                axes[idx].axis("off")
            
            plt.suptitle("Phase 3A: Stamp Removal Evidence Grid", fontsize=14, fontweight="bold", y=0.98)
            plt.tight_layout()
            plt.show()
            print(f"\n[OK] Evidence grid displayed ({len(evidence_images)} images)")
        else:
            # Fallback: simple before/after if grid images not found
            original = cv2.cvtColor(cv2.imread(INPUT_IMAGE), cv2.COLOR_BGR2RGB)
            cleaned = cv2.cvtColor(cv2.imread(CLEANED_IMAGE), cv2.COLOR_BGR2RGB)
            
            fig, axes = plt.subplots(1, 2, figsize=(16, 8))
            axes[0].imshow(original)
            axes[0].set_title("Before (Original with Seal)", fontsize=12, fontweight="bold")
            axes[0].axis("off")
            
            axes[1].imshow(cleaned)
            axes[1].set_title("After (Stamp Removed)", fontsize=12, fontweight="bold")
            axes[1].axis("off")
            
            plt.suptitle("Phase 3A: Stamp Removal Result", fontsize=14, fontweight="bold")
            plt.tight_layout()
            plt.show()
else:
    print(f"[WARN] Phase 3A had issues: {result_3a.get('error', 'Unknown error')}")
    print(f"[INFO] Continuing with original image for OCR")
    CLEANED_IMAGE = INPUT_IMAGE

## Cell 5: Phase 4 - OCR and Text Extraction

In [ ]:
PHASE_4_OUTPUT = str(OUTPUT_ROOT / "phase_4")

print("Running Phase 4: OCR and Text Extraction...")
print("[*] First run downloads EasyOCR Spanish model (~100MB, takes ~5 min)")
print("[*] Subsequent runs are much faster\n")

result_4 = run_phase_4(
    input_image_path=CLEANED_IMAGE,
    output_dir=PHASE_4_OUTPUT,
    language="es",
    confidence_threshold=0.6,
    levenshtein_threshold=2,
)

if result_4["success"]:
    OCR_JSON = result_4["output_json"]
    print(f"\n[OK] Phase 4 completed successfully")
    print(f"  Output JSON: {OCR_JSON}")
    print(f"  Text blocks detected: {result_4.get('text_blocks', 'N/A')}")
    print(f"  Average confidence: {result_4.get('confidence', 0):.2%}")
    print(f"  Words corrected: {result_4.get('words_corrected', 0)}")
    print(f"  Characters extracted: {result_4.get('text_length', 0):,}")

    # Display extracted text
    import json
    with open(OCR_JSON, encoding="utf-8") as f:
        ocr_data = json.load(f)
    
    extracted = ocr_data.get("texto_completo", "")
    print("\n" + "="*80)
    print("EXTRACTED TEXT (first 800 characters)")
    print("="*80)
    print(extracted[:800])
    if len(extracted) > 800:
        print(f"\n... [{len(extracted)-800} more characters]")
    print("="*80)
else:
    print(f"[ERROR] Phase 4 failed: {result_4.get('error', 'Unknown error')}")
    OCR_JSON = None

## Cell 6: Phase 4 - Color-Coded Detection Visualization

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

if OCR_JSON and Path(OCR_JSON).exists():
    print("[*] Creating color-coded OCR detection visualization...\n")
    
    # Load OCR JSON
    with open(OCR_JSON, encoding="utf-8") as f:
        ocr_data = json.load(f)
    
    # Load cleaned image
    img = cv2.imread(CLEANED_IMAGE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_vis = img_rgb.copy()
    
    # Extract blocks and confidence
    bloques = ocr_data.get("bloques", [])
    
    confidence_counts = {"high": 0, "medium": 0, "low": 0}
    
    # Draw bounding boxes with confidence-based coloring
    for bloque in bloques:
        coords = bloque.get("coordenadas", [])
        confianza = bloque.get("confianza", 0)
        texto = bloque.get("texto", "")
        
        if len(coords) == 4:
            # Confidence-based color coding
            if confianza >= 0.8:
                color = (0, 255, 0)  # GREEN - high confidence
                category = "high"
            elif confianza >= 0.5:
                color = (0, 255, 255)  # YELLOW - medium confidence
                category = "medium"
            else:
                color = (0, 0, 255)  # RED - low confidence
                category = "low"
            
            confidence_counts[category] += 1
            
            # Convert coordinates to numpy array for polylines
            pts = np.array(coords, dtype=np.int32)
            pts = pts.reshape((-1, 1, 2))
            
            # Draw polyline (bounding box)
            cv2.polylines(img_vis, [pts], True, color, 2)
            
            # Draw confidence text
            x, y = int(coords[0][0]), int(coords[0][1])
            conf_text = f"{confianza:.0%}"
            cv2.putText(img_vis, conf_text, (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    # Draw legend
    legend_y = 30
    cv2.putText(img_vis, "Confidence Legend:", (10, legend_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    legend_items = [
        ((0, 255, 0), f"High (>=80%): {confidence_counts['high']} blocks"),
        ((0, 255, 255), f"Medium (50-80%): {confidence_counts['medium']} blocks"),
        ((0, 0, 255), f"Low (<50%): {confidence_counts['low']} blocks"),
    ]
    
    for idx, (color, label) in enumerate(legend_items):
        y = legend_y + 30 + (idx * 25)
        cv2.rectangle(img_vis, (15, y-10), (30, y+5), color, -1)
        cv2.putText(img_vis, label, (40, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    
    # Display visualization
    plt.figure(figsize=(16, 12))
    plt.imshow(img_vis)
    plt.title(f"Phase 4: Color-Coded OCR Detections (Total blocks: {len(bloques)})", fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print(f"\n[OK] Detection visualization complete")
    print(f"  Total text blocks: {len(bloques)}")
    print(f"  High confidence (>=80%): {confidence_counts['high']} blocks")
    print(f"  Medium confidence (50-80%): {confidence_counts['medium']} blocks")
    print(f"  Low confidence (<50%): {confidence_counts['low']} blocks")
else:
    print("[INFO] Skipping detection visualization — OCR output not available")

## Cell 7: Consolidate Results + Metrics Dashboard

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

CONSOLIDATED_OUTPUT = str(OUTPUT_ROOT / "consolidated")

if OCR_JSON and Path(OCR_JSON).exists():
    print("Consolidating OCR results...\n")
    
    metrics = consolidate_ocr_results(
        ocr_json_path=OCR_JSON,
        output_dir=CONSOLIDATED_OUTPUT,
    )

    txt_file = Path(CONSOLIDATED_OUTPUT) / "ocr_extracted_text.txt"
    json_file = Path(CONSOLIDATED_OUTPUT) / "ocr_metrics.json"

    print(f"[OK] Results consolidated")
    print(f"\n[FILES GENERATED]")
    print(f"  1. {txt_file.name}")
    print(f"     Plain text format with character/word counts")
    print(f"     Location: {txt_file}")
    print(f"\n  2. {json_file.name}")
    print(f"     Comprehensive metrics in JSON")
    print(f"     Location: {json_file}")

    # Extract metrics
    tm = metrics.get("text_metrics", {})
    cm = metrics.get("ocr_confidence_metrics", {})
    pm = metrics.get("post_processing_metrics", {})

    # Load OCR JSON for confidence distribution
    with open(OCR_JSON, encoding="utf-8") as f:
        ocr_data = json.load(f)
    
    bloques = ocr_data.get("bloques", [])
    confidences = [b.get("confianza", 0) for b in bloques]
    
    # === METRICS DASHBOARD ===
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.25)
    
    # 1. Confidence histogram
    ax1 = fig.add_subplot(gs[0, 0])
    if confidences:
        ax1.hist(confidences, bins=5, color='skyblue', edgecolor='black', alpha=0.7)
        ax1.set_xlabel("Confidence Level", fontweight="bold")
        ax1.set_ylabel("Number of Blocks", fontweight="bold")
        ax1.set_title("OCR Confidence Distribution", fontweight="bold")
        ax1.grid(axis='y', alpha=0.3)
    
    # 2. Confidence pie chart
    ax2 = fig.add_subplot(gs[0, 1])
    high = sum(1 for c in confidences if c >= 0.8)
    medium = sum(1 for c in confidences if 0.5 <= c < 0.8)
    low = sum(1 for c in confidences if c < 0.5)
    sizes = [high, medium, low]
    labels = [f'High (>=80%)\n{high} blocks', f'Medium (50-80%)\n{medium} blocks', f'Low (<50%)\n{low} blocks']
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    if sum(sizes) > 0:
        ax2.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        ax2.set_title("Confidence Breakdown", fontweight="bold")
    
    # 3. Metrics table
    ax3 = fig.add_subplot(gs[1, :])
    ax3.axis('off')
    metrics_data = [
        ["Characters", f"{tm.get('total_characters', 0):,}"],
        ["Words", f"{tm.get('total_words', 0):,}"],
        ["Unique words", f"{tm.get('unique_words', 0):,}"],
        ["Text blocks detected", f"{tm.get('total_blocks', 0)}"],
        ["Avg confidence", f"{cm.get('average_percent', 0):.2f}%"],
        ["High confidence blocks", f"{cm.get('blocks_with_high_confidence', 0)}"],
        ["Words corrected (spell-check)", f"{pm.get('words_corrected', 0)}"],
        ["Named entities detected", f"{pm.get('entities_detected', 0)}"],
    ]
    table = ax3.table(cellText=metrics_data, colLabels=["Metric", "Value"],
                     cellLoc='left', loc='center', colWidths=[0.6, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    for i in range(len(metrics_data) + 1):
        table[(i, 0)].set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')
        table[(i, 1)].set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')
    table[(0, 0)].set_facecolor('#3498db')
    table[(0, 1)].set_facecolor('#3498db')
    table[(0, 0)].set_text_props(weight='bold', color='white')
    table[(0, 1)].set_text_props(weight='bold', color='white')
    
    # 4. Quality assessment
    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis('off')
    
    avg_conf = cm.get('average_percent', 0) / 100
    if avg_conf >= 0.85:
        quality = "EXCELLENT"
        color = '#2ecc71'
    elif avg_conf >= 0.75:
        quality = "VERY GOOD"
        color = '#3498db'
    elif avg_conf >= 0.6:
        quality = "GOOD"
        color = '#f39c12'
    else:
        quality = "NEEDS IMPROVEMENT"
        color = '#e74c3c'
    
    quality_text = f"OCR Quality: {quality} (Avg Confidence: {cm.get('average_percent', 0):.2f}%)"
    ax4.text(0.5, 0.5, quality_text, fontsize=16, fontweight='bold',
            ha='center', va='center', 
            bbox=dict(boxstyle='round,pad=0.8', facecolor=color, alpha=0.7, edgecolor='black', linewidth=2))
    
    plt.suptitle("Phase 4: OCR Metrics Dashboard", fontsize=14, fontweight="bold", y=0.995)
    plt.show()
    
    print(f"\n[METRICS SUMMARY]")
    print(f"  Characters:        {tm.get('total_characters', 0):,}")
    print(f"  Words:             {tm.get('total_words', 0):,}")
    print(f"  Unique words:      {tm.get('unique_words', 0):,}")
    print(f"  Text blocks:       {tm.get('total_blocks', 0)}")
    print(f"  Avg confidence:    {cm.get('average_percent', 0):.2f}%")
    print(f"  High confidence:   {cm.get('blocks_with_high_confidence', 0)} blocks (>=80%)")
    print(f"  Words corrected:   {pm.get('words_corrected', 0)}")
    print(f"  Entities detected: {pm.get('entities_detected', 0)}")
    print(f"\n  Quality Assessment: {quality}")
    
else:
    print("[INFO] Skipping consolidation — OCR output not available")

## Cell 8: Phase 5 - Evaluation (Optional, requires ground truth)

In [ ]:
from pathlib import Path

HAS_GROUND_TRUTH = False  # Set to True to provide ground truth file
GROUND_TRUTH_FILE = r"C:\path\to\ground_truth.txt"  # Update this path

if HAS_GROUND_TRUTH:
    txt_file = Path(CONSOLIDATED_OUTPUT) / "ocr_extracted_text.txt"

    if Path(GROUND_TRUTH_FILE).exists() and txt_file.exists():
        # Load texts
        with open(txt_file, encoding="utf-8") as f:
            ocr_lines = [l for l in f.read().split("\n")
                          if not l.startswith("=") and not l.startswith("Total ") and l.strip()]
            ocr_text = "\n".join(ocr_lines).strip()

        with open(GROUND_TRUTH_FILE, encoding="utf-8") as f:
            gt_text = f.read().strip()

        # Compute CER/WER
        try:
            from src.phase_5.levenshtein_calculator import LevenshteinCalculator
            lev = LevenshteinCalculator()
            
            ref_chars = list(gt_text.lower())
            hyp_chars = list(ocr_text.lower())
            cer = lev.calculate(ref_chars, hyp_chars) / max(len(ref_chars), 1)

            ref_words = gt_text.lower().split()
            hyp_words = ocr_text.lower().split()
            wer = lev.calculate(ref_words, hyp_words) / max(len(ref_words), 1)

            print(f"\n[EVALUATION RESULTS]")
            print(f"  CER (Char Error Rate):  {cer*100:6.2f}%  (lower is better)")
            print(f"  WER (Word Error Rate):  {wer*100:6.2f}%  (lower is better)")
            print(f"  Character Accuracy:     {(1-cer)*100:6.2f}%")
            print(f"  Word Accuracy:          {(1-wer)*100:6.2f}%")
            
            if cer < 0.05:
                quality = "EXCELLENT"
            elif cer < 0.1:
                quality = "VERY GOOD"
            elif cer < 0.2:
                quality = "GOOD"
            elif cer < 0.5:
                quality = "FAIR"
            else:
                quality = "NEEDS IMPROVEMENT"
            print(f"\n  Quality Assessment: {quality}")
            
        except ImportError:
            print("[INFO] LevenshteinCalculator not available — skipping eval")
    else:
        print(f"[ERROR] Ground truth file not found: {GROUND_TRUTH_FILE}")
        print(f"[INFO] Update GROUND_TRUTH_FILE in Cell 8 and re-run")
else:
    print("[INFO] Ground truth not provided — skipping evaluation")
    print("[*] To evaluate, set HAS_GROUND_TRUTH = True")
    print("[*] Update GROUND_TRUTH_FILE path to point to your ground truth .txt file")

## Cell 9: Summary and Output Locations

In [ ]:
from pathlib import Path

print("\n" + "="*80)
print("PIPELINE EXECUTION SUMMARY")
print("="*80)

print(f"\n[OUTPUT DIRECTORY]")
print(f"  Location: {OUTPUT_ROOT}")
print(f"\n[FILES GENERATED]")

output_files = [
    (OUTPUT_ROOT / "phase_3a", "Phase 3A outputs (stamp removal evidence)"),
    (OUTPUT_ROOT / "phase_4", "Phase 4 outputs (OCR detection JSON)"),
    (OUTPUT_ROOT / "consolidated" / "ocr_extracted_text.txt", "Extracted OCR text (plain text)"),
    (OUTPUT_ROOT / "consolidated" / "ocr_metrics.json", "OCR metrics and statistics (JSON)"),
]

for path, desc in output_files:
    if isinstance(path, Path) and path.exists():
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_str = f"{size} B"
            elif size < 1024*1024:
                size_str = f"{size/1024:.1f} KB"
            else:
                size_str = f"{size/(1024*1024):.1f} MB"
            print(f"  [OK] {path.name:35} ({size_str:>8}) - {desc}")
        else:
            file_count = len(list(path.glob("*")))
            print(f"  [OK] {path.name:35} ({file_count} files) - {desc}")
    else:
        print(f"  [ ] {path.name:35} (not yet generated) - {desc}")

print(f"\n[NEXT STEPS]")
print(f"  1. Check output files in: {OUTPUT_ROOT}")
print(f"  2. View ocr_extracted_text.txt for the extracted document text")
print(f"  3. View ocr_metrics.json for detailed confidence and processing statistics")
print(f"  4. To run evaluation with CER/WER metrics:")
print(f"     - Place ground truth file at a known location")
print(f"     - Update GROUND_TRUTH_FILE in Cell 8")
print(f"     - Set HAS_GROUND_TRUTH = True")
print(f"     - Re-run Cell 8")

print(f"\n" + "="*80)